In [161]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf

# Forklaring af variablerne:

### TFP = Total Factor Productivity

TFP måler hvor effektivt et land producerer, når man tager højde for:
- arbejdskraft
- kapital
- maskiner
- timer arbejdet

Hvor god økonomien er til at omsætte ressourcer til output.

Simpelt eksempel

Land A: 100 arbejdere + 10 maskiner → producerer 1000

Land B: 100 arbejdere + 10 maskiner → producerer 1500

Så har Land B højere TFP.

### HC – Human Capital

HC måler:
- uddannelse
- erfaring
- kompetencer
- viden
- arbejdsstyrkens kvalitet

En ingeniør, læge eller veluddannet arbejder bidrager typisk mere produktivt end ufaglært arbejdskraft.

### GDP per capita

GDP = BNP per indbygger

Måler gennemsnitlig økonomisk produktion pr person.

Jo højere GDP pc, jo rigere land typisk.

Et højt GDP kan også skyldes:
- stor ulighed
- få meget rige personer
- olieindtægter
- kapitalgevinster

gennemsnitlig velstand, ikke perfekt mål for almindelige borgeres økonomi.

### ODR – Old Dependency Ratio

Måler:

Antal ældre personer relativt til personer i arbejdsalderen.

Typisk:
- ældre = 65+
- arbejdsalder = 15-64

Formel:
𝑂𝐷𝑅 = (65+15 / 64) × 100

Eksempel:
Hvis ODR = 20

Så betyder det cirka:

20 ældre for hver 100 personer i arbejdsalderen.

Hvis ODR er høj:

Ja, så er der relativt flere ældre og færre arbejdende til at forsørge dem.

Det betyder IKKE at alle 15-64 arbejder.

Det er bare aldersgrupper.

Kort intuition

Høj ODR kan give:

færre arbejdere
højere pensionspres
mindre dynamik
lavere innovation (måske)

In [162]:
# Indlæs data
GDP = pd.read_csv("GDP.csv", skiprows=4)
ODR = pd.read_csv("age-dependency-ratio-old.csv")
TFP = pd.read_csv("total-factor-productivity.csv")
HC_DATA = pd.read_excel("pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']] # Relevante kolonner fra filen HC_DATA (pwt110)

Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [163]:
# Udvælger relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Beregner BNP per capita ud fra rgdpe og befolkning
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Udtrækker observationsåret 2002 og beholder de variable, der skal bruges som initiale forklarende variable i tværsnittet
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()

# Omdøber variablerne, så det tydeligt fremgår, at de er målt i 2002
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# Udtrækker TFP i slutåret 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()

# Omdøber TFP-variablen for 2020
tfp2020.columns = ['country','TFP2020']

# Merge datasæt

Datasættene er filtreret og indeholder kun de relevante kolonner. 

Benytter følgende formel:

Δ log(TFPi​) = log(TFP2020​) − log(TFP2002​)

som viser ændringen i log TFP fra periodens start (fx 2002) til slut (fx 2020)

In [164]:
# Merger 2002- og 2020-data, så hvert land får både initial og slut-TFP
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# Konstruerer long-difference outcome som ændringen i log(TFP) fra 2002 til 2020
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# Udtrækker old dependency ratio i startåret 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()

# Omdøber variablerne, så de matcher merge-strukturen
odr2002.columns = ['country', 'ODR2002']

# Sammenfletter ODR med tværsnittet, så hvert land får sin initiale ODR-værdi
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')

# Fjerner observationer med manglende værdier
Tvaersnit = Tvaersnit.dropna()

In [165]:
print(Tvaersnit.describe())

           HC2002     TFP2002     GDPpc2002     TFP2020   GrowthTFP  \
count  108.000000  108.000000    108.000000  108.000000  108.000000   
mean     2.500757    0.674189  18667.498333    0.637384   -0.014267   
std      0.659964    0.296350  17809.178627    0.220858    0.304781   
min      1.088122    0.167584    728.561351    0.200132   -1.073842   
25%      2.082968    0.403153   4843.455089    0.471157   -0.224303   
50%      2.576247    0.652114  11166.490440    0.625006   -0.045691   
75%      2.967072    0.883181  32387.904671    0.819311    0.183162   
max      3.598119    1.394810  84592.225755    1.236640    0.942964   

          ODR2002  
count  108.000000  
mean    12.640748  
std      7.701428  
min      1.583193  
25%      6.062487  
50%      8.882906  
75%     20.099692  
max     28.059470  


### Model 1: Growth_TFP = α + β ODR2002 + ε

Har lande med højere ODR i 2002 lavere TFP-vækst 2002–2020?

In [166]:
model1 = smf.ols("GrowthTFP ~ ODR2002", data=Tvaersnit).fit()
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.069
Model:                            OLS   Adj. R-squared:                  0.060
Method:                 Least Squares   F-statistic:                     7.814
Date:                Tue, 21 Apr 2026   Prob (F-statistic):            0.00616
Time:                        11:34:58   Log-Likelihood:                -20.581
No. Observations:                 108   AIC:                             45.16
Df Residuals:                     106   BIC:                             50.53
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1168      0.055      2.130      0.0

Negativ og statistisk signifikant sammenhæng mellem ODR i 2002 og TFP-vækst 2002–2020.

ODR2002 = -0.0104 
- Hvis et land havde 1 point højere old dependency ratio i 2002, var TFP-væksten over perioden i gennemsnit ca. 0.010 lavere.

Signifikans
p = 0.006
- Signifikant på 1%-niveau - Så dette er ikke bare tilfældig støj.

R² = 0.069

- Forklarer ca. 7% af variationen.

### Konklussion 

Baseline-regressionen (Model 1) viser en negativ sammenhæng mellem initial old dependency ratio (ODR2002) og efterfølgende vækst i total factor productivity (GrowthTFP). Koefficienten på ODR2002 er -0.0104 og statistisk signifikant (p = 0.006), hvilket indikerer, at lande med relativt flere ældre i forhold til befolkningen i arbejdsalderen i 2002 oplevede lavere TFP-vækst i perioden 2002–2020.

### Model 2: Growth_TFP = α + βODR2002 + γ HC2002 + ε

Påvirker aldring stadig TFP-vækst, når vi tager højde for human capital?

In [167]:
model2 = smf.ols("GrowthTFP ~ ODR2002 + HC2002", data=Tvaersnit).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.074
Model:                            OLS   Adj. R-squared:                  0.056
Method:                 Least Squares   F-statistic:                     4.191
Date:                Tue, 21 Apr 2026   Prob (F-statistic):             0.0177
Time:                        11:34:58   Log-Likelihood:                -20.274
No. Observations:                 108   AIC:                             46.55
Df Residuals:                     105   BIC:                             54.60
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0321      0.123      0.262      0.7

ODR2002 = -0.0135

før

- -0.0104

Nu:

- -0.0135

Så når vi holder humankapital konstant, bliver aldringseffekten endnu tydeligere.

Et land med 1 point højere ODR i 2002 havde ca. 0.0135 lavere TFP-vækst 2002–2020, alt andet lige.

p = 0.016

- signifikant på 5%-niveau.

HC2002

- Koefficient = 0.0498

- p = 0.441

Ikke signifikant.

- Vi finder ikke selvstændig evidens for, at initial humankapital forklarer TFP-vækst i denne simple model.

Efter kontrol for human capital forbliver koefficienten på ODR negativ og statistisk signifikant. Dette indikerer, at sammenhængen mellem aldring og lavere produktivitetsvækst ikke alene kan forklares af forskelle i humankapital.

### Model 3: GrowthTFP = α + β ODR200 2 γ HC2002 + δ log(GDPpc2002) + ε

Påvirker aldring stadig produktivitetsvækst, når vi sammenligner lande med samme humankapital og samme udviklingsniveau ved startåret?

Er aldring stadig vigtig, når to lande starter lige rige?

In [168]:
model3 = smf.ols("GrowthTFP ~ ODR2002 + HC2002 + np.log(GDPpc2002)", data=Tvaersnit).fit()
print(model3.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.177
Model:                            OLS   Adj. R-squared:                  0.153
Method:                 Least Squares   F-statistic:                     7.433
Date:                Tue, 21 Apr 2026   Prob (F-statistic):           0.000147
Time:                        11:34:58   Log-Likelihood:                -13.931
No. Observations:                 108   AIC:                             35.86
Df Residuals:                     104   BIC:                             46.59
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.8431      0.25

ODR2002
- Koefficient = -0.0086

p = 0.114

- Ikke signifikant ved 5%.

Når vi tager højde for udviklingsniveau og human capital: aldringseffekten svækkes statistisk

HC2002
- Koefficient = +0.1819

p = 0.012

Lande med højere human capital i 2002 oplevede højere TFP-vækst.

log(GDPpc2002)
- Koefficient = -0.1298

p < 0.001

Rige lande voksede langsommere end fattige lande.


En del af ODR-effekten hænger sammen med, at ældre lande ofte også er rige og modne økonomier.

### Model 4: Δlog(TFP​) = α + β ODR_i,2002​ + γ HC_i,2002 ​+ δ TFP_i,2002 ​+ ε_i​

Hænger initial aldring sammen med efterfølgende TFP-vækst, når vi også tager højde for initial humankapital og initialt produktivitetsniveau?

In [169]:
model4 = smf.ols("GrowthTFP ~ ODR2002 + HC2002 + TFP2002", data=Tvaersnit).fit()
print(model4.summary())

                            OLS Regression Results                            
Dep. Variable:              GrowthTFP   R-squared:                       0.390
Model:                            OLS   Adj. R-squared:                  0.373
Method:                 Least Squares   F-statistic:                     22.19
Date:                Tue, 21 Apr 2026   Prob (F-statistic):           3.49e-11
Time:                        11:34:58   Log-Likelihood:                 2.2940
No. Observations:                 108   AIC:                             3.412
Df Residuals:                     104   BIC:                             14.14
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2031      0.103      1.980      0.0

- ODR2002

p = 0.920

Når vi tager højde for landets start-produktivitetsniveau, forklarer aldring ikke yderligere TFP-vækst.

- HC2002

Positiv og signifikant

Mere humankapital ved start hænger sammen med højere efterfølgende vækst.

- TFP2002

-0.7501 og ekstremt signifikant

Lande med højt initialt TFP voksede langsommere bagefter.

Lande med lavt initialt TFP voksede hurtigere.

- R² sprang til 0.390

Fordi initial TFP er en meget stærk forklaringsvariabel.

Det er normalt i growth regressions.